# Lab 05 - Enriquecimento de Dados em Tempo Real (Stream-Static Join)
Neste laboratório, vamos cruzar um fluxo contínuo de eventos com uma tabela estática de referência de negócio.

**Objetivos:**
1. Criar um DataFrame estático de lookup/dimensão.
2. Realizar o join entre Streaming e Static DataFrames com Schema explícito.
3. Consultar os dados enriquecidos via Spark SQL.

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# 1. Criando dados estáticos de referência (Catálogo / Dimensão de Lookup)
static_data = [
    ("Open", "Abertura de Sessão", "Alta Prioridade"),
    ("Close", "Encerramento de Sessão", "Média Prioridade")
]
df_static = spark.createDataFrame(static_data, ["action", "descricao", "prioridade"])
display(df_static)

In [0]:
# 2. Configurando a Ingestão em Streaming com Schema explícito
inputPath = "/databricks-datasets/structured-streaming/events/"
jsonSchema = StructType([ 
    StructField("time", TimestampType(), True), 
    StructField("action", StringType(), True) 
])

df_streaming = (spark.readStream
                .schema(jsonSchema)
                .option("maxFilesPerTrigger", 1)
                .json(inputPath))

# 3. Realizando o Stream-Static Join em Tempo Real
df_enriched = df_streaming.join(df_static, "action")

In [0]:
# 4. Inicializando a Query de Enriquecimento

# Parar eventuais queries ativas anteriores
for stream in spark.streams.active:
    stream.stop()

# Limpeza de checkpoint do Lab 05
checkpoint_path = "/Volumes/workspace/default/checkpoint/lab05_checkpoint"
dbutils.fs.rm(checkpoint_path, True)

query = (df_enriched.writeStream
         .format("memory")
         .queryName("eventos_enriquecidos")
         .outputMode("append")
         .option("checkpointLocation", checkpoint_path)
         .trigger(availableNow=True)
         .start())

In [0]:
# 5. Consulta dos dados enriquecidos via SQL
display(spark.sql("SELECT time, action, descricao, prioridade FROM eventos_enriquecidos ORDER BY time DESC LIMIT 20"))

In [0]:
# 6. Encerramento gracioso da query
print(f"Status da query antes de parar: {query.status}")
query.stop()
print("Query de enriquecimento encerrada com sucesso.")